# Optimization backends

Capacity models build a solver-independent `Problem` containing an objective, parameter layout, bounds, and capacity constraints. `Optimizer` then translates that problem to SciPy, CVXPY, or PYMOO.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, mean_squared_error

from capacities_ml_fin.ml.models import ChoquetClassifier, ChoquetRegressor
from capacities_ml_fin.ml.optimization import KAdditivity, Solver

np.set_printoptions(precision=5, suppress=True)


## 1. Choosing a backend

| Backend | Best suited to | Main limitation |
|---|---|---|
| CVXPY | Convex objectives with a symbolic form | Cannot represent the direct 0-1 classification loss |
| SciPy/SLSQP | Smooth constrained problems, including local non-convex optimization | A local solution may depend on initialization |
| PYMOO/GA | Discontinuous objectives such as 0-1 loss | More function evaluations and finite bounds are required |

The model determines which backend is mathematically appropriate; solver names are not interchangeable in every estimator.


## 2. Convex regression: SciPy versus CVXPY

Both backends solve the same constrained least-squares problem. The predictors are already normalized to `[0, 1]`, and a 2-additive representation keeps singleton and pairwise Möbius terms.


In [ ]:
rng = np.random.default_rng(17)
X_regression = rng.uniform(0.0, 1.0, size=(70, 3))
y_regression = (
    0.30
    + 0.20 * X_regression[:, 0]
    + 0.25 * X_regression[:, 1]
    + 0.15 * X_regression[:, 2]
    + 0.25 * np.minimum(X_regression[:, 0], X_regression[:, 1])
    + 0.15 * np.minimum(X_regression[:, 1], X_regression[:, 2])
    + rng.normal(0.0, 0.015, X_regression.shape[0])
)
sparsity = KAdditivity(order=2)

regressors = {
    "SciPy": ChoquetRegressor(
        sparsity=sparsity,
        solver=Solver.SCIPY,
        solver_options={"method": "SLSQP", "options": {"maxiter": 2000, "ftol": 1e-12}},
    ),
    "CVXPY": ChoquetRegressor(
        sparsity=sparsity,
        solver=Solver.CVXPY,
        solver_options={"warm_start": True, "verbose": False},
    ),
}

for model in regressors.values():
    model.fit(X_regression, y_regression)


### Compare solutions and constraint diagnostics

For this convex problem, both methods should reach nearly the same objective. Constraint violation should be close to zero.


In [ ]:
def maximum_constraint_violation(model):
    parameters = model.result_.parameters
    constraints = model.problem_.constraints
    violations = [constraints.bounds.maximum_violation(parameters)]
    violations.extend(item.maximum_violation(parameters) for item in constraints.linear_constraints)
    violations.extend(item.maximum_violation(parameters) for item in constraints.nonlinear_constraints)
    return max(violations)


comparison = {}
for name, model in regressors.items():
    prediction = model.predict(X_regression)
    comparison[name] = {
        "success": model.result_.success,
        "backend": model.result_.solver_name,
        "MSE": mean_squared_error(y_regression, prediction),
        "objective": model.result_.objective_value,
        "runtime_seconds": model.result_.runtime_seconds,
        "iterations": model.result_.n_iterations,
        "max_constraint_violation": maximum_constraint_violation(model),
    }

display(pd.DataFrame(comparison))
print("Maximum absolute prediction difference:", np.max(np.abs(
    regressors["SciPy"].predict(X_regression) - regressors["CVXPY"].predict(X_regression)
)))


### Compare fitted Möbius coefficients

Small numerical differences are expected because the backends use different algorithms and stopping criteria.


In [ ]:
def readable_coefficients(model):
    return {
        "{" + ", ".join(coalition) + "}": value
        for coalition, value in model.capacity_.to_named_dict().items()
    }


display(
    pd.DataFrame(
        {name: readable_coefficients(model) for name, model in regressors.items()}
    )
)


## 3. Discontinuous classification with PYMOO

The direct Choquet classifier minimizes the number of classification errors. This objective is piecewise constant and discontinuous, so the genetic backend searches a population of capacities and thresholds.


In [ ]:
rng = np.random.default_rng(29)
X_classification = rng.uniform(0.0, 1.0, size=(80, 3))
latent_score = (
    0.10 * X_classification[:, 0]
    + 0.10 * X_classification[:, 1]
    + 0.10 * X_classification[:, 2]
    + 0.55 * np.minimum(X_classification[:, 0], X_classification[:, 1])
    + 0.15 * np.minimum(X_classification[:, 1], X_classification[:, 2])
)
y_classification = (latent_score >= 0.48).astype(int)

classifier = ChoquetClassifier(
    sparsity=KAdditivity(order=2),
    solver=Solver.PYMOO,
    solver_options={
        "population_size": 80,
        "n_generations": 100,
        "seed": 29,
        "verbose": False,
        "equality_tolerance": 1e-4,
    },
).fit(X_classification, y_classification)

classification_prediction = classifier.predict(X_classification)


### Genetic-search result

`objective_value` is the number of training errors because the estimator uses the summed 0-1 loss. Accuracy presents the same result on a `[0, 1]` scale.


In [ ]:
classification_summary = pd.Series(
    {
        "success": classifier.result_.success,
        "backend": classifier.result_.solver_name,
        "training errors": int(np.sum(classification_prediction != y_classification)),
        "objective": classifier.result_.objective_value,
        "accuracy": accuracy_score(y_classification, classification_prediction),
        "threshold": classifier.threshold_,
        "runtime_seconds": classifier.result_.runtime_seconds,
        "generations": classifier.result_.n_iterations,
        "max_constraint_violation": maximum_constraint_violation(classifier),
    }
)
display(classification_summary)
display(pd.Series(classifier.feature_scales_, index=classifier.get_feature_names_out(), name="feature scale"))
display(pd.Series(readable_coefficients(classifier), name="fitted Möbius coefficient"))


## 4. Practical rule

- Use **CVXPY** when the complete formulation is convex and has a symbolic translation.
- Use **SciPy/SLSQP** for smooth constrained models such as Choquistic regression or the Choquet autoregression.
- Use **PYMOO** for the direct classifier's discontinuous 0-1 objective.

Always inspect `success`, the objective, runtime, and maximum constraint violation. Predictive metrics must still be evaluated on held-out data; optimizer success only means that the numerical problem was solved acceptably.
